In [0]:
customers_df = spark.read.csv(
    "/Volumes/workspace/default/olist/olist/olist_customers_dataset.csv",
    header=True,
    inferSchema=True
)

In [0]:
orders_df = spark.read.csv(
    "/Volumes/workspace/default/olist/olist/olist_orders_dataset.csv",
    header=True,
    inferSchema=True
)

order_item_df = spark.read.csv(
    "/Volumes/workspace/default/olist/olist/olist_order_items_dataset.csv",
    header=True,
    inferSchema=True
)

payments_df = spark.read.csv(
    "/Volumes/workspace/default/olist/olist/olist_order_payments_dataset.csv",
    header=True,
    inferSchema=True
)

reviews_df = spark.read.csv(
    "/Volumes/workspace/default/olist/olist/olist_order_reviews_dataset.csv",
    header=True,
    inferSchema=True
)

products_df = spark.read.csv(
    "/Volumes/workspace/default/olist/olist/olist_products_dataset.csv",
    header=True,
    inferSchema=True
)

sellers_df = spark.read.csv(
    "/Volumes/workspace/default/olist/olist/olist_sellers_dataset.csv",
    header=True,
    inferSchema=True
)

geolocation_df = spark.read.csv(
    "/Volumes/workspace/default/olist/olist/olist_geolocation_dataset.csv",
    header=True,
    inferSchema=True
)

category_translation_df = spark.read.csv(
    "/Volumes/workspace/default/olist/olist/product_category_name_translation.csv",
    header=True,
    inferSchema=True
)

In [0]:
print("Customers:", customers_df.count())
print("Orders:", orders_df.count())
print("Order Items:", order_item_df.count())
print("Payments:", payments_df.count())
print("Reviews:", reviews_df.count())
print("Products:", products_df.count())
print("Sellers:", sellers_df.count())
print("Geolocation:", geolocation_df.count())
print("Category Translation:", category_translation_df.count())

Customers: 99441
Orders: 99441
Order Items: 112650
Payments: 103886
Reviews: 104162
Products: 32951
Sellers: 3095
Geolocation: 1000163
Category Translation: 71


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/olist/olist"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/olist/olist/olist_customers_dataset.csv,olist_customers_dataset.csv,9033957,1781537432000
dbfs:/Volumes/workspace/default/olist/olist/olist_geolocation_dataset.csv,olist_geolocation_dataset.csv,61273883,1781537455000
dbfs:/Volumes/workspace/default/olist/olist/olist_order_items_dataset.csv,olist_order_items_dataset.csv,15438671,1781537438000
dbfs:/Volumes/workspace/default/olist/olist/olist_order_payments_dataset.csv,olist_order_payments_dataset.csv,5777138,1781537429000
dbfs:/Volumes/workspace/default/olist/olist/olist_order_reviews_dataset.csv,olist_order_reviews_dataset.csv,14451670,1781537438000
dbfs:/Volumes/workspace/default/olist/olist/olist_orders_dataset.csv,olist_orders_dataset.csv,17654914,1781537440000
dbfs:/Volumes/workspace/default/olist/olist/olist_products_dataset.csv,olist_products_dataset.csv,2379446,1781537425000
dbfs:/Volumes/workspace/default/olist/olist/olist_sellers_dataset.csv,olist_sellers_dataset.csv,174703,1781537422000
dbfs:/Volumes/workspace/default/olist/olist/product_category_name_translation.csv,product_category_name_translation.csv,2613,1781537422000


In [0]:
customers_df.show(5)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 5 rows


In [0]:
# ================================================
# MODULE 1 - EXPLORATION (Continuing)
# ================================================

from pyspark.sql.functions import col, count, when, datediff, sum

# 1. Check Schema
print("=== CUSTOMERS SCHEMA ===")
customers_df.printSchema()

print("=== ORDERS SCHEMA ===")
orders_df.printSchema()

=== CUSTOMERS SCHEMA ===
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

=== ORDERS SCHEMA ===
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [0]:
# 2. Check Null Values in all dataframes
def check_nulls(df, name):
    print(f"\n=== NULL VALUES IN {name} ===")
    df.select([count(when(col(c).isNull(),1))
        .alias(c) for c in df.columns]).show()

check_nulls(customers_df, 'CUSTOMERS')
check_nulls(orders_df, 'ORDERS')
check_nulls(order_item_df, 'ORDER ITEMS')
check_nulls(payments_df, 'PAYMENTS')


=== NULL VALUES IN CUSTOMERS ===
+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+


=== NULL VALUES IN ORDERS ===
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+--------------------------

In [0]:
# 3. Customer Distribution by State
print("=== CUSTOMER DISTRIBUTION BY STATE ===")
customers_df.groupBy('customer_state')\
    .count()\
    .orderBy('count', ascending=False)\
    .show()

# 4. Order Status Distribution
print("=== ORDER STATUS DISTRIBUTION ===")
orders_df.groupBy('order_status')\
    .count()\
    .orderBy('count', ascending=False)\
    .show()

# 5. Payment Type Distribution
print("=== PAYMENT TYPE DISTRIBUTION ===")
payments_df.groupBy('payment_type')\
    .count()\
    .orderBy('count', ascending=False)\
    .show()

=== CUSTOMER DISTRIBUTION BY STATE ===
+--------------+-----+
|customer_state|count|
+--------------+-----+
|            SP|41746|
|            RJ|12852|
|            MG|11635|
|            RS| 5466|
|            PR| 5045|
|            SC| 3637|
|            BA| 3380|
|            DF| 2140|
|            ES| 2033|
|            GO| 2020|
|            PE| 1652|
|            CE| 1336|
|            PA|  975|
|            MT|  907|
|            MA|  747|
|            MS|  715|
|            PB|  536|
|            PI|  495|
|            RN|  485|
|            AL|  413|
+--------------+-----+
only showing top 20 rows
=== ORDER STATUS DISTRIBUTION ===
+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|96478|
|     shipped| 1107|
|    canceled|  625|
| unavailable|  609|
|    invoiced|  314|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+

=== PAYMENT TYPE DISTRIBUTION ===
+------------+-----+
|payment_type|count|
+------------+-----+


In [0]:
# 6. Top Selling Products by Revenue
print("=== TOP 20 PRODUCTS BY REVENUE ===")
order_item_df.groupBy('product_id')\
    .agg(sum('price').alias('total_sales'))\
    .orderBy('total_sales', ascending=False)\
    .show(20)

# 7. Delivery Time Analysis
print("=== DELIVERY TIME ANALYSIS ===")
delivery_df = orders_df.select(
    'order_id',
    'order_purchase_timestamp',
    'order_delivered_customer_date'
)

delivery_detail_df = delivery_df.withColumn(
    'delivery_time_days',
    datediff(
        col('order_delivered_customer_date'),
        col('order_purchase_timestamp')
    )
)

delivery_detail_df\
    .orderBy('delivery_time_days', ascending=False)\
    .show(10)

print("\n MODULE 1 COMPLETE!")

=== TOP 20 PRODUCTS BY REVENUE ===
+--------------------+------------------+
|          product_id|       total_sales|
+--------------------+------------------+
|bb50f2e236e5eea01...|           63885.0|
|6cdd53843498f9289...| 54730.19999999998|
|d6160fb7873f18409...|48899.340000000004|
|d1c427060a0f73f6b...| 47214.50999999997|
|99a4788cb24856965...|43025.559999999976|
|3dd2a17168ec895c7...| 41082.59999999999|
|25c38557cf793876c...|          38907.32|
|5f504b3a1c75b73d6...|           37733.9|
|53b36df67ebb7c415...|          37683.42|
|aca2eb7d00ea1a7b8...|           37608.9|
|e0d64dcfaa3b6db5c...|          31786.82|
|d285360f29ac7fd97...|          31623.81|
|7a10781637204d8d1...|30467.499999999996|
|f1c7f353075ce59d8...|          29997.36|
|f819f0c84a64f02d3...|          29024.48|
|588531f8ec37e7d5f...|28291.989999999998|
|422879e10f4668299...| 26577.22000000002|
|16c4e87b98a9370a9...|           25034.0|
|5a848e4ab52fd5445...|24229.029999999984|
|a62e25e09e05e6faf...|           24051.0|

In [0]:
# ================================================
# MODULE 2 - DATA CLEANING AND TRANSFORMATION
# ================================================

from pyspark.sql.functions import *
from pyspark.ml.feature import Imputer

# --- STEP 1: Handle Missing Values ---

# Drop rows where critical columns are null
orders_df_cleaned = orders_df.na.drop(
    subset=['order_id','customer_id','order_status']
)

# Fill missing delivery date with placeholder
orders_df_cleaned = orders_df_cleaned.fillna(
    {'order_delivered_customer_date': '9999-12-31'}
)

print("=== ORDERS AFTER CLEANING ===")
orders_df_cleaned.select([count(when(col(c).isNull(),1))
    .alias(c) for c in orders_df_cleaned.columns]).show()

=== ORDERS AFTER CLEANING ===
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [0]:
# --- STEP 2: Impute Missing Payment Values ---

# Create artificial nulls to demonstrate imputation
payments_df_with_null = payments_df.withColumn(
    'payment_value',
    when(col('payment_value') != 99.33, col('payment_value'))
    .otherwise(lit(None))
)

# Impute using median strategy
imputer = Imputer(
    inputCols=['payment_value'],
    outputCols=['payment_value_imputed']
).setStrategy('median')

payments_df_cleaned = imputer.fit(payments_df_with_null)\
    .transform(payments_df_with_null)

print("=== PAYMENTS AFTER IMPUTATION ===")
payments_df_cleaned.select(
    'payment_value',
    'payment_value_imputed'
).show(5)

=== PAYMENTS AFTER IMPUTATION ===
+-------------+---------------------+
|payment_value|payment_value_imputed|
+-------------+---------------------+
|         NULL|                100.0|
|        24.39|                24.39|
|        65.71|                65.71|
|       107.78|               107.78|
|       128.45|               128.45|
+-------------+---------------------+
only showing top 5 rows


In [0]:
# --- STEP 3: Standardize Payment Type Names ---

payments_df_cleaned = payments_df_cleaned.withColumn(
    'payment_type',
    when(col('payment_type') == 'boleto', 'Bank Transfer')
    .when(col('payment_type') == 'credit_card', 'Credit Card')
    .when(col('payment_type') == 'debit_card', 'Debit Card')
    .otherwise('Other')
)

print("=== PAYMENT TYPES AFTER STANDARDIZATION ===")
payments_df_cleaned.groupBy('payment_type')\
    .count()\
    .orderBy('count', ascending=False)\
    .show()

=== PAYMENT TYPES AFTER STANDARDIZATION ===
+-------------+-----+
| payment_type|count|
+-------------+-----+
|  Credit Card|76795|
|Bank Transfer|19784|
|        Other| 5778|
|   Debit Card| 1529|
+-------------+-----+



In [0]:
# --- STEP 4: Fix Data Types ---

# Convert zip code from integer to string
customers_df_cleaned = customers_df.withColumn(
    'customer_zip_code_prefix',
    col('customer_zip_code_prefix').cast('string')
)

# Convert purchase timestamp to date only
orders_df_cleaned = orders_df_cleaned.withColumn(
    'order_purchase_timestamp',
    to_date(col('order_purchase_timestamp'))
)

print("=== CUSTOMERS SCHEMA AFTER FIX ===")
customers_df_cleaned.printSchema()

print("=== ORDERS SCHEMA AFTER FIX ===")
orders_df_cleaned.printSchema()

=== CUSTOMERS SCHEMA AFTER FIX ===
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

=== ORDERS SCHEMA AFTER FIX ===
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: date (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [0]:
# --- STEP 7: Create New Feature - Product Size Category ---

products_df_cleaned = products_df.withColumn(
    'product_size_category',
    when(col('product_weight_g') < 500, 'Small')
    .when(col('product_weight_g').between(500, 2000), 'Medium')
    .otherwise('Large')
)

print("=== PRODUCTS WITH SIZE CATEGORY ===")
products_df_cleaned.select(
    'product_id',
    'product_weight_g',
    'product_size_category'
).show(10)

print("\n✅ MODULE 2 COMPLETE!")

=== PRODUCTS WITH SIZE CATEGORY ===
+--------------------+----------------+---------------------+
|          product_id|product_weight_g|product_size_category|
+--------------------+----------------+---------------------+
|1e9e8ef04dbcff454...|             225|                Small|
|3aa071139cb16b67c...|            1000|               Medium|
|96bd76ec8810374ed...|             154|                Small|
|cef67bcfe19066a93...|             371|                Small|
|9dc1a7de274444849...|             625|               Medium|
|41d3672d4792049fa...|             200|                Small|
|732bd381ad09e530f...|           18350|                Large|
|2548af3e6e77a690c...|             900|               Medium|
|37cc742be07708b53...|             400|                Small|
|8c92109888e8cdf9d...|             600|               Medium|
+--------------------+----------------+---------------------+
only showing top 10 rows

✅ MODULE 2 COMPLETE!


In [0]:
# --- STEP 5: Remove Duplicates ---

customers_df_cleaned = customers_df_cleaned\
    .dropDuplicates(['customer_id'])

print(f"Customers before dedup: {customers_df.count()}")
print(f"Customers after dedup: {customers_df_cleaned.count()}")

Customers before dedup: 99441
Customers after dedup: 99441


In [0]:
# --- STEP 6: Handle Price Outliers ---

# Find price boundaries using quantiles
quantiles = order_item_df.approxQuantile(
    'price', [0.01, 0.99], 0.0
)
low_cutoff = quantiles[0]
high_cutoff = quantiles[1]

print(f"Low cutoff: {low_cutoff}")
print(f"High cutoff: {high_cutoff}")

# Filter out outliers
order_item_df_cleaned = order_item_df.filter(
    (col('price') >= low_cutoff) & 
    (col('price') <= high_cutoff)
)

print(f"Order items before: {order_item_df.count()}")
print(f"Order items after: {order_item_df_cleaned.count()}")

Low cutoff: 9.99
High cutoff: 890.0
Order items before: 112650
Order items after: 110453


In [0]:
# ================================================
# MODULE 3 - DATA INTEGRATION AND AGGREGATION
# ================================================

from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Note: Caching is not supported in Serverless mode
# In production clusters, we would use .cache() here
# orders_df.cache()
# customers_df_cleaned.cache()
# order_item_df_cleaned.cache()

print("Module 3 Started!")
print("Note: Running on Serverless - caching skipped")

Module 3 Started!
Note: Running on Serverless - caching skipped


In [0]:
# --- STEP 2: Join all datasets together ---

# Start with orders + items
orders_items_df = orders_df_cleaned\
    .join(order_item_df_cleaned, 'order_id', 'inner')

# Add products
orders_items_products_df = orders_items_df\
    .join(products_df_cleaned, 'product_id', 'inner')

# Add sellers (broadcast - sellers table is small)
orders_items_products_sellers_df = orders_items_products_df\
    .join(broadcast(sellers_df), 'seller_id', 'inner')

# Add customers
full_orders_df = orders_items_products_sellers_df\
    .join(customers_df_cleaned, 'customer_id', 'inner')

# Add geolocation (broadcast - used for lookup)
full_orders_df = full_orders_df\
    .join(
        broadcast(geolocation_df),
        full_orders_df.customer_zip_code_prefix == 
        geolocation_df.geolocation_zip_code_prefix,
        'left'
    )

# Add reviews
full_orders_df = full_orders_df\
    .join(broadcast(reviews_df), 'order_id', 'left')

# Add payments
full_orders_df = full_orders_df\
    .join(payments_df_cleaned, 'order_id', 'left')

# Cache the final joined dataframe
#full_orders_df.cache(), not supporting

print("All datasets joined successfully!")
print(f"Total rows: {full_orders_df.count()}")
print(f"Total columns: {len(full_orders_df.columns)}")

All datasets joined successfully!
Total rows: 17705373
Total columns: 46


In [0]:
# --- STEP 3: KEY AGGREGATIONS ---

# 3a. Total Revenue per Seller
print("=== TOP 10 SELLERS BY REVENUE ===")
seller_revenue_df = full_orders_df\
    .groupBy('seller_id')\
    .agg(sum('price').alias('total_revenue'))\
    .orderBy('total_revenue', ascending=False)
seller_revenue_df.show(10)

# 3b. Total Orders per Customer
print("=== TOP 10 CUSTOMERS BY ORDER COUNT ===")
customer_order_count_df = full_orders_df\
    .groupBy('customer_id')\
    .agg(count('order_id').alias('total_orders'))\
    .orderBy('total_orders', ascending=False)
customer_order_count_df.show(10)

# 3c. Average Review Score per Seller
print("=== TOP 10 SELLERS BY REVIEW SCORE ===")
seller_review_df = full_orders_df\
    .groupBy('seller_id')\
    .agg(avg('review_score').alias('avg_review_score'))\
    .orderBy('avg_review_score', ascending=False)
seller_review_df.show(10)

# 3d. Top 10 Most Sold Products
print("=== TOP 10 MOST SOLD PRODUCTS ===")
top_products_df = full_orders_df\
    .groupBy('product_id')\
    .agg(count('order_id').alias('total_sold'))\
    .orderBy('total_sold', ascending=False)\
    .limit(10)
top_products_df.show()

=== TOP 10 SELLERS BY REVENUE ===
+--------------------+--------------------+
|           seller_id|       total_revenue|
+--------------------+--------------------+
|4869f7a5dfa277a7d...| 3.605861821000105E7|
|4a3ca9315b744ce9f...|   3.3759570839997E7|
|7c67e1448b00f6e96...|3.2282321790015433E7|
|da8622b14eb17ae28...|  2.98569569299862E7|
|fa1c13f2614d7b5c4...|2.6013136649997156E7|
|1025f0e2d44d7041d...|2.2937518519999195E7|
|955fee9216a65b617...|2.0964410669996638E7|
|46dc3b2cc0980fb8e...|2.0583881680002622E7|
|7a67c85e85bb2ce85...|2.0312794890006024E7|
|620c87c171fb2a6dd...| 2.011983959999356E7|
+--------------------+--------------------+
only showing top 10 rows
=== TOP 10 CUSTOMERS BY ORDER COUNT ===
+--------------------+------------+
|         customer_id|total_orders|
+--------------------+------------+
|351e40989da90e704...|       11427|
|50920f8cd0681fd86...|       10752|
|9b43e2a62de9bab3a...|        8556|
|270c23a11d024a44c...|        8001|
|5c87184371002d49e...|        687

In [0]:
# --- STEP 4: CUSTOMER SPENDING ANALYSIS ---

# Calculate total spending and AOV per customer
customer_spending_df = full_orders_df\
    .groupBy('customer_id')\
    .agg(
        count('order_id').alias('total_orders'),
        sum('price').alias('total_spent'),
        round(avg('price'), 2).alias('AOV')
    )\
    .orderBy('total_spent', ascending=False)

print("=== TOP 10 CUSTOMERS BY SPENDING ===")
customer_spending_df.show(10)

# Customer Segmentation based on AOV
customer_spending_df = customer_spending_df.withColumn(
    'customer_segment',
    when(col('AOV') >= 1200, 'High-Value')
    .when(
        (col('AOV') < 1200) & (col('AOV') >= 700),
        'Medium-Value'
    )
    .otherwise('Low-Value')
)

print("=== CUSTOMER SEGMENTS ===")
customer_spending_df.groupBy('customer_segment')\
    .count()\
    .show()

=== TOP 10 CUSTOMERS BY SPENDING ===
+--------------------+------------+------------------+------+
|         customer_id|total_orders|       total_spent|   AOV|
+--------------------+------------+------------------+------+
|63b964e79dee32a35...|        6072|         2501664.0| 412.0|
|1ff773612ab8934db...|        5820|1658641.7999999512|284.99|
|de832e8dbb1f588a4...|        2190|1584990.5999999817|723.74|
|d72181923840c8895...|        2721|1488114.8999999566| 546.9|
|904805ce22729b9b7...|        3306|         1319094.0| 399.0|
|1dbc055ccab23ed89...|        4535| 1314696.499999974| 289.9|
|9af2372a1e4934027...|        2726|1070091.3000000538|392.55|
|351e40989da90e704...|       11427| 982607.7299999113| 85.99|
|f034ceb6936919a21...|        1102|          935598.0| 849.0|
|8d00bb042dfdfc4c8...|        3960| 934164.0000000617| 235.9|
+--------------------+------------+------------------+------+
only showing top 10 rows
=== CUSTOMER SEGMENTS ===
+----------------+-----+
|customer_segment|c

In [0]:
# --- STEP 5: ADVANCED ENRICHMENT ---

# Add order status flags
full_orders_df = full_orders_df\
    .withColumn(
        'is_delivered',
        when(col('order_status') == 'delivered', 1)
        .otherwise(0)
    )\
    .withColumn(
        'is_canceled',
        when(col('order_status') == 'canceled', 1)
        .otherwise(0)
    )

# Add order revenue (price + freight)
full_orders_df = full_orders_df\
    .withColumn(
        'order_revenue',
        col('price') + col('freight_value')
    )

# Add hour of day
full_orders_df = full_orders_df\
    .withColumn(
        'hour_of_day',
        hour(col('order_purchase_timestamp'))
    )

# Add weekday vs weekend
full_orders_df = full_orders_df\
    .withColumn(
        'order_day_type',
        when(
            dayofweek('order_purchase_timestamp').isin(1, 7),
            'Weekend'
        ).otherwise('Weekday')
    )

# Join customer segment back to main df
full_orders_df = full_orders_df.join(
    customer_spending_df.select(
        'customer_id', 'customer_segment'
    ),
    'customer_id',
    'left'
)

print("=== FINAL DATAFRAME SCHEMA ===")
full_orders_df.printSchema()
print("\n MODULE 3 COMPLETE!")

=== FINAL DATAFRAME SCHEMA ===
root
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: date (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- produc

In [0]:
# ================================================
# MODULE 4 - SPARK OPTIMIZATION
# ================================================

print("=== SPARK OPTIMIZATION MODULE ===")
print("""
In Production Clusters, we optimize Spark with:

1. MEMORY SETTINGS:
   - spark.executor.memory = 6g
     (Each worker gets 6GB RAM)
   - spark.driver.memory = 4g  
     (Main controller gets 4GB RAM)

2. PARALLELISM SETTINGS:
   - spark.sql.shuffle.partitions = 64
     (Split data into 64 parts during joins)
   - spark.default.parallelism = 64
     (Run 64 tasks simultaneously)

3. ADAPTIVE QUERY EXECUTION:
   - spark.sql.adaptive.enabled = true
     (Spark auto-optimizes at runtime)

4. JOIN OPTIMIZATION:
   - spark.sql.autoBroadcastJoinThreshold = 20MB
     (Auto broadcast tables smaller than 20MB)

NOTE: On Databricks Serverless, these are 
managed automatically by Databricks!
""")

print("Spark Configuration Overview Complete!")

=== SPARK OPTIMIZATION MODULE ===

In Production Clusters, we optimize Spark with:

1. MEMORY SETTINGS:
   - spark.executor.memory = 6g
     (Each worker gets 6GB RAM)
   - spark.driver.memory = 4g  
     (Main controller gets 4GB RAM)

2. PARALLELISM SETTINGS:
   - spark.sql.shuffle.partitions = 64
     (Split data into 64 parts during joins)
   - spark.default.parallelism = 64
     (Run 64 tasks simultaneously)

3. ADAPTIVE QUERY EXECUTION:
   - spark.sql.adaptive.enabled = true
     (Spark auto-optimizes at runtime)

4. JOIN OPTIMIZATION:
   - spark.sql.autoBroadcastJoinThreshold = 20MB
     (Auto broadcast tables smaller than 20MB)

NOTE: On Databricks Serverless, these are 
managed automatically by Databricks!

Spark Configuration Overview Complete!


In [0]:
# --- STEP 2: JOIN OPTIMIZATION STRATEGIES ---

print("=== JOIN OPTIMIZATION STRATEGIES ===")
print("""
Three main join strategies in Spark:

1. BROADCAST JOIN:
   - Send small dataframe to ALL worker nodes
   - Avoids shuffling data across network
   - Best for: Small tables (sellers, products)
   - Code: df.join(broadcast(small_df), 'key')

2. SORT MERGE JOIN:
   - Sort both dataframes by join key first
   - Then merge them together
   - Best for: Two large dataframes
   - Code: df1.sortWithinPartitions('key')
           .join(df2.sortWithinPartitions('key'))

3. BUCKET JOIN:
   - Pre-partition both dataframes by same key
   - Reduces shuffling for repeated joins
   - Best for: Large dataframes joined repeatedly
   - Code: df.repartition(10, 'key')
""")

# Demonstrate Broadcast Join (works on serverless)
print("=== DEMONSTRATING BROADCAST JOIN ===")
broadcast_join_df = orders_df_cleaned.join(
    broadcast(sellers_df),
    orders_df_cleaned.order_id == order_item_df_cleaned.order_id,
    'left'
)
print("Broadcast join executed successfully!")
print("This is the most commonly used optimization in production!")

=== JOIN OPTIMIZATION STRATEGIES ===

Three main join strategies in Spark:

1. BROADCAST JOIN:
   - Send small dataframe to ALL worker nodes
   - Avoids shuffling data across network
   - Best for: Small tables (sellers, products)
   - Code: df.join(broadcast(small_df), 'key')

2. SORT MERGE JOIN:
   - Sort both dataframes by join key first
   - Then merge them together
   - Best for: Two large dataframes
   - Code: df1.sortWithinPartitions('key')
           .join(df2.sortWithinPartitions('key'))

3. BUCKET JOIN:
   - Pre-partition both dataframes by same key
   - Reduces shuffling for repeated joins
   - Best for: Large dataframes joined repeatedly
   - Code: df.repartition(10, 'key')

=== DEMONSTRATING BROADCAST JOIN ===
Broadcast join executed successfully!
This is the most commonly used optimization in production!


In [0]:
# --- STEP 3: WINDOW FUNCTIONS ---

print("=== WINDOW FUNCTION: RANK TOP PRODUCTS PER SELLER ===")

window_spec = Window\
    .partitionBy('seller_id')\
    .orderBy(desc('price'))

top_seller_products_df = full_orders_df\
    .withColumn('rank', rank().over(window_spec))\
    .filter(col('rank') <= 3)

top_seller_products_df\
    .select('seller_id', 'product_id', 'price', 'rank')\
    .show(10)

print("Window Functions complete!")

=== WINDOW FUNCTION: RANK TOP PRODUCTS PER SELLER ===
+--------------------+--------------------+------+----+
|           seller_id|          product_id| price|rank|
+--------------------+--------------------+------+----+
|004c9cd9d87a3c30c...|6ef8c8d7d06c5dca9...|259.99|   1|
|004c9cd9d87a3c30c...|6ef8c8d7d06c5dca9...|259.99|   1|
|004c9cd9d87a3c30c...|6ef8c8d7d06c5dca9...|259.99|   1|
|004c9cd9d87a3c30c...|6ef8c8d7d06c5dca9...|259.99|   1|
|004c9cd9d87a3c30c...|6ef8c8d7d06c5dca9...|259.99|   1|
|004c9cd9d87a3c30c...|6ef8c8d7d06c5dca9...|259.99|   1|
|004c9cd9d87a3c30c...|6ef8c8d7d06c5dca9...|259.99|   1|
|004c9cd9d87a3c30c...|6ef8c8d7d06c5dca9...|259.99|   1|
|004c9cd9d87a3c30c...|6ef8c8d7d06c5dca9...|259.99|   1|
|004c9cd9d87a3c30c...|6ef8c8d7d06c5dca9...|259.99|   1|
+--------------------+--------------------+------+----+
only showing top 10 rows
Window Functions complete!


In [0]:
# --- STEP 4: ADVANCED AGGREGATIONS ---

# Seller Performance Metrics
print("=== SELLER PERFORMANCE METRICS ===")
seller_performance_df = full_orders_df\
    .groupBy('seller_id')\
    .agg(
        count('order_id').alias('total_orders'),
        sum('price').alias('total_revenue'),
        round(avg('review_score'), 2)\
            .alias('avg_review_score'),
        round(stddev('price'), 2)\
            .alias('price_variability')
    )\
    .orderBy('total_revenue', ascending=False)

seller_performance_df.show(10)

# Product Popularity Metrics
print("=== PRODUCT POPULARITY METRICS ===")
product_metrics_df = full_orders_df\
    .groupBy('product_id')\
    .agg(
        count('order_id').alias('total_sales'),
        sum('price').alias('total_revenue'),
        round(avg('price'), 2).alias('avg_price'),
        round(stddev('price'), 2)\
            .alias('price_volatility')
    )\
    .orderBy('total_sales', ascending=False)

product_metrics_df\
    .select(
        'product_id',
        'total_sales',
        'total_revenue',
        'avg_price',
        'price_volatility'
    ).show(10)

print("\nMODULE 4 COMPLETE!")

=== SELLER PERFORMANCE METRICS ===
+--------------------+------------+--------------------+----------------+-----------------+
|           seller_id|total_orders|       total_revenue|avg_review_score|price_variability|
+--------------------+------------+--------------------+----------------+-----------------+
|4869f7a5dfa277a7d...|      184498| 3.605861821000105E7|            4.09|            110.6|
|4a3ca9315b744ce9f...|      330661|   3.3759570839997E7|            3.77|            59.37|
|7c67e1448b00f6e96...|      233306|3.2282321790015433E7|            3.42|            50.39|
|da8622b14eb17ae28...|      264361|  2.98569569299862E7|            3.98|            72.91|
|fa1c13f2614d7b5c4...|       84723|2.6013136649997156E7|            4.38|           204.32|
|1025f0e2d44d7041d...|      229587|2.2937518519999195E7|            3.89|             84.3|
|955fee9216a65b617...|      232364|2.0964410669996638E7|            4.04|            84.94|
|46dc3b2cc0980fb8e...|       89587|2.05838816

In [0]:
# ================================================
# MODULE 5 - DATA SERVING LAYER
# ================================================

# --- WHAT IS DATA SERVING LAYER? ---
# After all cleaning and processing, we need to SAVE
# our final data so others can use it
# Think of it like: after cooking food, you serve it on a plate!

# We save in 3 formats:
# 1. Parquet - for data engineers and analysts (fast, compressed)
# 2. Delta Table - for SQL users to query
# 3. CSV - for business users (opens in Excel)

# --- STEP 1: Save as Parquet ---
print("=== SAVING AS PARQUET FORMAT ===")

full_orders_df.write\
    .mode('overwrite')\
    .parquet('/Volumes/workspace/default/olist/processed/full_orders')

print("Saved as Parquet successfully!")

=== SAVING AS PARQUET FORMAT ===
Saved as Parquet successfully!


In [0]:
# --- STEP 2: Save as Delta Table ---
print("=== SAVING AS DELTA TABLE ===")

full_orders_df.write\
    .mode('overwrite')\
    .saveAsTable('olist_full_orders')

print("Saved as Delta Table successfully!")
print("SQL users can now query: SELECT * FROM olist_full_orders")

=== SAVING AS DELTA TABLE ===
Saved as Delta Table successfully!
SQL users can now query: SELECT * FROM olist_full_orders


In [0]:
# --- STEP 3: Save as CSV ---
print("=== SAVING AS CSV FORMAT ===")

full_orders_df.write\
    .mode('overwrite')\
    .option('header', 'true')\
    .csv('/Volumes/workspace/default/olist/processed/full_orders_csv')

print("Saved as CSV successfully!")

=== SAVING AS CSV FORMAT ===
Saved as CSV successfully!


In [0]:
# ================================================
# PROJECT SUMMARY
# ================================================

print("""
╔════════════════════════════════════════════════╗
║     OLIST E-COMMERCE DATA ENGINEERING PROJECT  ║
║              PROJECT SUMMARY                   ║
╠════════════════════════════════════════════════╣
║                                                ║
║  Dataset: Brazilian E-Commerce (Olist/Kaggle)  ║
║  Platform: Databricks Serverless               ║
║  Language: PySpark                             ║
║                                                ║
║  MODULE 1: Data Ingestion & Exploration        ║
║  - Loaded 9 CSV files into Spark DataFrames   ║
║  - Explored schemas, nulls, distributions     ║
║  - Delivery time analysis                     ║
║                                                ║
║  MODULE 2: Data Cleaning & Transformation      ║
║  - Handled null values (drop/fill/impute)     ║
║  - Removed duplicates                         ║
║  - Removed price outliers (percentile method) ║
║  - Standardized payment type names            ║
║  - Created product size categories            ║
║                                                ║
║  MODULE 3: Data Integration & Aggregation      ║
║  - Joined all 9 datasets into master df       ║
║  - Used broadcast joins for optimization      ║
║  - Customer segmentation (High/Med/Low)       ║
║  - Seller performance metrics                 ║
║  - Product popularity metrics                 ║
║  - Hour of day & weekday analysis             ║
║                                                ║
║  MODULE 4: Spark Optimization                  ║
║  - Understood Spark configurations            ║
║  - Implemented 3 join strategies              ║
║  - Window functions & rankings                ║
║  - Advanced aggregations                      ║
║                                                ║
║  MODULE 5: Data Serving Layer                  ║
║  - Saved as Parquet (for engineers)           ║
║  - Saved as Delta Table (for SQL users)       ║
║  - Saved as CSV (for business users)          ║
║                                                ║
╚════════════════════════════════════════════════╝
""")


╔════════════════════════════════════════════════╗
║     OLIST E-COMMERCE DATA ENGINEERING PROJECT  ║
║              PROJECT SUMMARY                   ║
╠════════════════════════════════════════════════╣
║                                                ║
║  Dataset: Brazilian E-Commerce (Olist/Kaggle)  ║
║  Platform: Databricks Serverless               ║
║  Language: PySpark                             ║
║                                                ║
║  MODULE 1: Data Ingestion & Exploration        ║
║  - Loaded 9 CSV files into Spark DataFrames   ║
║  - Explored schemas, nulls, distributions     ║
║  - Delivery time analysis                     ║
║                                                ║
║  MODULE 2: Data Cleaning & Transformation      ║
║  - Handled null values (drop/fill/impute)     ║
║  - Removed duplicates                         ║
║  - Removed price outliers (percentile method) ║
║  - Standardized payment type names            ║
║  - Created product size categories 